# Analytical Base Table -- Build & Descriptive Profile

Merged from `Analytical_Base_Table.ipynb` (builds the ABT from the three raw
CSVs) and `ABT_Descriptive.ipynb` (profiles it). The descriptive section now
runs directly against the table just built in memory, instead of re-reading
`Group2_ABT.csv` from disk -- that file is never produced by the build
section, so the two notebooks couldn't previously be run back-to-back
without it existing already from somewhere else. See the chat reply below
this for why, and what to check if `Group2_ABT.csv` is actually a distinct,
further-processed file you still need elsewhere.


## Path setup (added)

Defines the data directory once, matching this notebook's location (two
folders below the project root, same convention as the original
`../../data/...` paths).


In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]  # two levels up, matching this notebook's location
DATA_DIR = PROJECT_ROOT / "data"


## **Imports & Loading CSV**


In [2]:
import pandas as pd

air_quality_path = str(DATA_DIR / "ph_air_quality_full.csv")
area_density_path = str(DATA_DIR / "Population_LandArea_Density_Province.csv")
station_province_path = str(DATA_DIR / "stations_city_province-stations.csv")


df_air_quality = pd.read_csv(air_quality_path)
df_area_density = pd.read_csv(area_density_path)
df_station_province = pd.read_csv(station_province_path)

## **Data Inspection**


In [3]:
print(df_air_quality.head(10))
print(df_area_density.head(10))
print(df_station_province.head(10))

     station  location_id  sensor_id parameter      lat       lon  \
0  313_Cubao      5963700   14249763       pm1  14.6227  121.0528   
1  313_Cubao      5963700   14249763       pm1  14.6227  121.0528   
2  313_Cubao      5963700   14249763       pm1  14.6227  121.0528   
3  313_Cubao      5963700   14249763       pm1  14.6227  121.0528   
4  313_Cubao      5963700   14249763       pm1  14.6227  121.0528   
5  313_Cubao      5963700   14249763       pm1  14.6227  121.0528   
6  313_Cubao      5963700   14249763       pm1  14.6227  121.0528   
7  313_Cubao      5963700   14249763       pm1  14.6227  121.0528   
8  313_Cubao      5963700   14249764      pm25  14.6227  121.0528   
9  313_Cubao      5963700   14249764      pm25  14.6227  121.0528   

                datetime_utc      value   unit  
0  2025-09-30 07:00:00+00:00   9.001000  µg/m³  
1  2025-09-30 22:00:00+00:00  34.000000  µg/m³  
2  2025-09-30 23:00:00+00:00  27.805894  µg/m³  
3  2025-10-01 00:00:00+00:00  20.776083  µg/

In [4]:
for col in df_air_quality.columns:
    print(f'Unique values for {col}: {df_air_quality[col].unique()}')
    print("-" * 100)

Unique values for station: <ArrowStringArray>
[                                                                              '313_Cubao',
                                                                     '3S Center Bagbaguin',
                                                                  '3S Center Paso De Blas',
                                                                         '3S Center Ugong',
                                                                    'ADB Courtyard (Demo)',
                                                                                   'ASMPH',
                                                                        'Along Shaw Blvd.',
                                                                             'Anda Circle',
                                               'Aurora Boulevard (Tramo) - Andrews Avenue',
                                                                              'Ayala Ave.',
                                  

In [5]:
for col in df_area_density.columns:
    print(f'Unique values for {col}: {df_area_density[col].unique()}')
    print("-" * 100)

Unique values for Region and Province: <ArrowStringArray>
[                                    'NCR',
                                   ' Abra',
                               '  Apayao ',
                             '  Benguet i',
                                '  Ifugao',
                             '  Kalinga  ',
                     '  Mountain Province',
                          '  Ilocos Norte',
                            '  Ilocos Sur',
                              '  La Union',
                            '  Pangasinan',
                               '  Batanes',
                               '  Cagayan',
                               '  Isabela',
                        '  Nueva Vizcaya ',
                               '  Quirino',
                               '  Aurora ',
                                '  Bataan',
                              '  Bulacan ',
                           '  Nueva Ecija',
                            '  Pampanga i',
                  

In [6]:
for col in df_station_province.columns:
    print(f'Unique values for {col}: {df_station_province[col].unique()}')
    print("-" * 100)

Unique values for station: <ArrowStringArray>
[                                                                              '313_Cubao',
                                                                     '3S Center Bagbaguin',
                                                                  '3S Center Paso De Blas',
                                                                         '3S Center Ugong',
                                                                    'ADB Courtyard (Demo)',
                                                                                   'ASMPH',
                                                                        'Along Shaw Blvd.',
                                                                             'Anda Circle',
                                               'Aurora Boulevard (Tramo) - Andrews Avenue',
                                                                              'Ayala Ave.',
                                  

In [7]:
print(df_air_quality.columns)
print(df_area_density.columns)
print(df_station_province.columns)

Index(['station', 'location_id', 'sensor_id', 'parameter', 'lat', 'lon',
       'datetime_utc', 'value', 'unit'],
      dtype='str')
Index(['Region and Province', 'Population', 'Land area (sq. km.) a1/a2/a3',
       'Density (persons/ sq km)'],
      dtype='str')
Index(['station', 'lat', 'lon', 'city', 'province'], dtype='str')


Problem 1:
Formatting of Provinces in `df_area_density` does not exactly match that of `df_station_province`

Implemented Solution for Problem 1:
1) Extract the necessary values needed in `df_stations_province`
2) Extract the equivalent values into a new file `df_area_density_prod`
3) Modify only `df_area_density_prod`
4) Merge into `Analytical Base Table`


## **Analytical Base Table**

In [8]:
province_list = df_station_province["province"].unique()
for province in province_list:
    province.strip()
print(province_list)


<ArrowStringArray>
[              'NCR', 'Negros Occidental',            'Bataan',
             'Rizal',           'Benguet',            'Iloilo',
            'Laguna',   'Camarines Norte',          'Pampanga',
     'Davao del Sur',              'Cebu',     'Eastern Samar',
            'Cavite',             'Capiz']
Length: 14, dtype: str


In [9]:
df_area_density_prod = df_area_density.copy()

df_area_density_prod['Region and Province'] = df_area_density_prod['Region and Province'].str.strip()
print(df_area_density_prod['Region and Province'].unique())

<ArrowStringArray>
[                                  'NCR',
                                  'Abra',
                                'Apayao',
                             'Benguet i',
                                'Ifugao',
                               'Kalinga',
                     'Mountain Province',
                          'Ilocos Norte',
                            'Ilocos Sur',
                              'La Union',
                            'Pangasinan',
                               'Batanes',
                               'Cagayan',
                               'Isabela',
                         'Nueva Vizcaya',
                               'Quirino',
                                'Aurora',
                                'Bataan',
                               'Bulacan',
                           'Nueva Ecija',
                            'Pampanga i',
                                'Tarlac',
                            'Zambales i',
               

In [10]:
df_area_density_prod['Region and Province'] = (
    df_area_density_prod['Region and Province']
    .str.replace(r'\s*\([^)]*\)', '', regex=True)
    .str.replace(r'\s+[0-9,i]+$', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

print(df_area_density_prod['Region and Province'].unique())

<ArrowStringArray>
[                  'NCR',                  'Abra',                'Apayao',
               'Benguet',                'Ifugao',               'Kalinga',
     'Mountain Province',          'Ilocos Norte',            'Ilocos Sur',
              'La Union',            'Pangasinan',               'Batanes',
               'Cagayan',               'Isabela',         'Nueva Vizcaya',
               'Quirino',                'Aurora',                'Bataan',
               'Bulacan',           'Nueva Ecija',              'Pampanga',
                'Tarlac',              'Zambales',              'Batangas',
                'Cavite',                'Laguna',                'Quezon',
                 'Rizal',            'Marinduque',    'Occidental Mindoro',
      'Oriental Mindoro',               'Palawan',               'Romblon',
                 'Albay',       'Camarines Norte',         'Camarines Sur',
           'Catanduanes',               'Masbate',              'Sors

In [11]:
df_air_quality = df_air_quality[df_air_quality['parameter']=="pm25"].copy()

In [12]:
analytical_base_table = pd.merge( df_air_quality,
                                  df_station_province,
                                  on= 'station'
                                  )

analytical_base_table.rename(columns={'value': 'pm25 in µg/m^3'}, inplace=True)
analytical_base_table.drop(columns=['location_id',
                                    'sensor_id',
                                    'lon_x',
                                    'lat_y',
                                    'parameter',
                                    'unit'], inplace=True)

In [13]:
for col in analytical_base_table.columns:
    print(f'Unique values for {col}: {analytical_base_table[col].unique()}')
    print("-" * 100)

Unique values for station: <ArrowStringArray>
[                                                                              '313_Cubao',
                                                                     '3S Center Bagbaguin',
                                                                  '3S Center Paso De Blas',
                                                                         '3S Center Ugong',
                                                                    'ADB Courtyard (Demo)',
                                                                                   'ASMPH',
                                                                        'Along Shaw Blvd.',
                                                                             'Anda Circle',
                                               'Aurora Boulevard (Tramo) - Andrews Avenue',
                                                                              'Ayala Ave.',
                                  

In [14]:
analytical_base_table = pd.merge(analytical_base_table,
                                 df_area_density_prod,
                                 left_on = 'province',
                                 right_on =  'Region and Province',
                                 )

analytical_base_table.drop(columns=['Region and Province'], inplace=True)
analytical_base_table.rename(columns={'Land area (sq. km.) a1/a2/a3': 'area_in_sq.km', 'Density (persons/ sq km)': 'density_persons/sqkm', 'Population': 'population', 'lat_x': 'lat', 'lon_y': 'lon' }, inplace=True)

In [15]:
print(analytical_base_table.columns
      )
print(analytical_base_table.head(10))

Index(['station', 'lat', 'datetime_utc', 'pm25 in µg/m^3', 'lon', 'city',
       'province', 'population', 'area_in_sq.km', 'density_persons/sqkm'],
      dtype='str')
               station       lat               datetime_utc  pm25 in µg/m^3  \
0            313_Cubao  14.62270  2025-09-30 07:00:00+00:00       19.652000   
1            313_Cubao  14.62270  2025-09-30 22:00:00+00:00       52.500000   
2            313_Cubao  14.62270  2025-09-30 23:00:00+00:00       41.965681   
3            313_Cubao  14.62270  2025-10-01 00:00:00+00:00       30.948667   
4            313_Cubao  14.62270  2025-10-01 01:00:00+00:00       30.113639   
5            313_Cubao  14.62270  2025-10-01 02:00:00+00:00       21.787083   
6            313_Cubao  14.62270  2025-10-01 03:00:00+00:00        7.477500   
7            313_Cubao  14.62270  2025-10-01 04:00:00+00:00        0.033333   
8  3S Center Bagbaguin  14.71405  2025-06-19 18:01:39+00:00       11.120000   
9  3S Center Bagbaguin  14.71405  2025-06-

In [16]:
analytical_base_table.to_csv(str(DATA_DIR / "Group2_ABT.csv"), index=False)

In [17]:
analytical_base_table.to_parquet(str(DATA_DIR / "Group2_ABT.parquet"), index=False)

---
## Descriptive Profile of the Analytical Base Table

The cells below are from `ABT_Descriptive.ipynb`. They originally re-read
the table from `Group2_ABT.csv` / `.parquet` on disk -- a different filename
than what the build section above actually produces
(`analytical_base_table.csv` / `.parquet`). Since this is now one notebook
in one kernel, we just alias the table already in memory instead:


In [18]:
import numpy as np  # NB2 also needs this; pandas and Path are already imported above

abt = analytical_base_table  # use the table just built, instead of re-reading a different file from disk


## **Null Statistics**


In [19]:
null_counts = abt.isnull().sum()
null_percent = (abt.isnull().sum() / len(abt)) * 100

null_stats = pd.DataFrame(
    {
        "Feature": abt.columns,
        "Null Count": null_counts.values,
        "Null Percentage (%)": null_percent.round(2).values,
    }
).sort_values(by="Null Count", ascending=False)

print("Null Statistics")
null_stats

Null Statistics


,Feature,Null Count,Null Percentage (%)
0,station,0,0.0
1,lat,0,0.0
2,datetime_utc,0,0.0
3,pm25 in µg/m^3,0,0.0
4,lon,0,0.0
5,city,0,0.0
6,province,0,0.0
7,population,0,0.0
8,area_in_sq.km,0,0.0
9,density_persons/sqkm,0,0.0


## **Numerical Feature Statistics**


In [20]:
numeric_stats = abt.describe(include=[np.number],percentiles=[0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]).T
print("\nNumerical Feature Statistics")
numeric_stats


Numerical Feature Statistics


,count,mean,std,min,5%,10%,25%,50%,75%,90%,95%,max
lat,1027171.0,14.297758,1.167600,7.143752,10.73537,14.37853,14.54570,14.58455,14.64443,14.71405,14.769440,16.49742
pm25 in µg/m^3,1027171.0,19.414052,28.260038,0.000000,2.03000,3.87000,7.80000,14.80000,24.08000,34.28000,44.089866,987.89000
lon,1027171.0,121.167812,0.778866,120.490300,120.59750,120.75050,120.98362,121.01877,121.05675,121.09803,122.548820,125.64968


## **Categorical Feature Statistics**


In [21]:
categorical_stats = abt.describe(include=["object", "category", "str"]).T
print("\nCategorical Feature Statistics")
categorical_stats


Categorical Feature Statistics


,count,unique,top,freq
station,1027171,97,3S Center Bagbaguin,17000
datetime_utc,1027171,897758,2026-09-04 01:00:00+00:00,20
city,1027171,37,Manila,103551
province,1027171,14,NCR,821222
population,1027171,14,"14,001,751",821222
area_in_sq.km,1027171,14,619.54,821222
density_persons/sqkm,1027171,14,"21,765",821222


## **Grouped Statistics by Province**


In [22]:
percentiles = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
quantile_funcs = [
    (f"p{int(p*100):02d}", (lambda q: lambda x: x.quantile(q))(p))
    for p in percentiles
]

print("\nGrouped Statistics by Province")
grouped_stats = abt.groupby("province")[["lat", "lon", "pm25 in µg/m^3"]].agg(["mean","min","max"] + quantile_funcs).T
grouped_stats


Grouped Statistics by Province


province                 Bataan     Benguet  Camarines Norte       Capiz  \
lat            mean   14.625148   16.497420        14.115700   11.430320   
               min    14.436000   16.497420        14.115700   11.430320   
               max    14.769440   16.497420        14.115700   11.430320   
               p05    14.515000   16.497420        14.115700   11.430320   
               p10    14.515000   16.497420        14.115700   11.430320   
               p25    14.515000   16.497420        14.115700   11.430320   
               p50    14.550700   16.497420        14.115700   11.430320   
               p75    14.677590   16.497420        14.115700   11.430320   
               p90    14.769440   16.497420        14.115700   11.430320   
               p95    14.769440   16.497420        14.115700   11.430320   
lon            mean  120.565661  120.653413       122.955100  122.926229   
               min   120.490300  120.653413       122.955100  122.926229   
               max   120.609000  120.653413       122.955100  122.926229   
               p05   120.518200  120.653413       122.955100  122.926229   
               p10   120.518200  120.653413       122.955100  122.926229   
               p25   120.518200  120.653413       122.955100  122.926229   
               p50   120.542780  120.653413       122.955100  122.926229   
               p75   120.597500  120.653413       122.955100  122.926229   
               p90   120.609000  120.653413       122.955100  122.926229   
               p95   120.609000  120.653413       122.955100  122.926229   
pm25 in µg/m^3 mean   10.622870   16.794275        31.434669   32.040528   
               min     0.000000    0.000000         0.100000    0.000000   
               max   144.970000  858.000000       125.700000  537.100000   
               p05     1.470000    0.100000         8.090000    2.300000   
               p10     2.550000    0.500000        11.500000    4.600000   
               p25     5.110000    2.640719        22.450000   12.000000   
               p50     8.840000    9.245312        31.400000   24.700000   
               p75    13.780000   25.800000        37.950000   38.300000   
               p90    20.290000   41.770967        44.720000   61.200000   
               p95    25.970000   51.400000        53.940000   88.875000   

province                 Cavite        Cebu  Davao del Sur  Eastern Samar  \
lat            mean   14.143177   10.326450       7.143752      10.998140   
               min    14.143177   10.326450       7.143752      10.998140   
               max    14.143177   10.326450       7.143752      10.998140   
               p05    14.143177   10.326450       7.143752      10.998140   
               p10    14.143177   10.326450       7.143752      10.998140   
               p25    14.143177   10.326450       7.143752      10.998140   
               p50    14.143177   10.326450       7.143752      10.998140   
               p75    14.143177   10.326450       7.143752      10.998140   
               p90    14.143177   10.326450       7.143752      10.998140   
               p95    14.143177   10.326450       7.143752      10.998140   
lon            mean  120.984941  123.978710     125.625054     125.649680   
               min   120.984941  123.978710     125.625054     125.649680   
               max   120.984941  123.978710     125.625054     125.649680   
               p05   120.984941  123.978710     125.625054     125.649680   
               p10   120.984941  123.978710     125.625054     125.649680   
               p25   120.984941  123.978710     125.625054     125.649680   
               p50   120.984941  123.978710     125.625054     125.649680   
               p75   120.984941  123.978710     125.625054     125.649680   
               p90   120.984941  123.978710     125.625054     125.649680   
               p95   120.984941  123.978710     125.625054     125.649680   
pm25 in µg/m^3 mean   16.1

## **Unique Values for Province**


In [23]:
abt["province"].unique()

<ArrowStringArray>
[              'NCR', 'Negros Occidental',            'Bataan',
             'Rizal',           'Benguet',            'Iloilo',
            'Laguna',   'Camarines Norte',          'Pampanga',
     'Davao del Sur',              'Cebu',     'Eastern Samar',
            'Cavite',             'Capiz']
Length: 14, dtype: str

# Initial Observations and Notable Findings

1. An overwhelming majority of sensors are located within the NCR province.
2. Only stations in Negros Oriental province provide pm10 data, thus it was excluded due to the limited number of observations.
3. Not all Philippine provinces are represented by air quality stations, a total of 14 provinces have coverage for air quality indicators.
4. Some stations have inconsistent reporting periods.